# 面试题：熔断与降级怎样保护 Agent 系统？

可复述答案：熔断器将连续下游失败变成本地快速拒绝。closed 正常调用，错误窗口超过阈值转 open，冷却后用少量 half-open 探测恢复。降级必须诚实地返回缓存时间、排队、草稿或人工升级，不能伪造新鲜成功。按工具、租户和错误类型分桶统计，避免一个租户的 4xx 封住全局服务。

## 真实案例

物流查询服务连续 5xx 时，客服 Agent 若继续调用会拖慢退款和订单服务。六个响应事件演示 closed、open、half-open 与缓存降级。

## 基线

基线无论失败多少次都直接调用物流服务。

## 结果解读

手写状态机输出请求、当前状态、动作和失败计数。

## 失败案例

把业务 404 与依赖 5xx 一起计数会造成错误熔断。

In [1]:
events = [{'id':'B01','response':500,'cache':'昨晚 22:00 已揽收'}, {'id':'B02','response':503,'cache':'昨晚 22:00 已揽收'}, {'id':'B03','response':500,'cache':'昨晚 22:00 已揽收'}, {'id':'B04','response':200,'cache':'昨晚 22:00 已揽收'}, {'id':'B05','response':200,'cache':'今天 09:00 派送中'}, {'id':'B06','response':404,'cache':'无缓存'}]  # 构造六个物流服务响应与可降级缓存。
print('物流事件输入:', events)  # 输出故障流和缓存内容。
print('教学说明：500/503 代表依赖故障，404 是业务结果而不是可熔断的基础设施故障。')  # 说明错误分类规则。

物流事件输入: [{'id': 'B01', 'response': 500, 'cache': '昨晚 22:00 已揽收'}, {'id': 'B02', 'response': 503, 'cache': '昨晚 22:00 已揽收'}, {'id': 'B03', 'response': 500, 'cache': '昨晚 22:00 已揽收'}, {'id': 'B04', 'response': 200, 'cache': '昨晚 22:00 已揽收'}, {'id': 'B05', 'response': 200, 'cache': '今天 09:00 派送中'}, {'id': 'B06', 'response': 404, 'cache': '无缓存'}]
教学说明：500/503 代表依赖故障，404 是业务结果而不是可熔断的基础设施故障。


In [2]:
baseline_calls = [row['id'] for row in events]  # 构造始终直接调用下游的基线调用清单。
print('无熔断基线调用:', baseline_calls)  # 输出故障期间仍压向下游的全部请求。
print('基线问题：连续失败也占用连接和重试预算，影响无关业务。')  # 解释没有局部隔离的级联风险。

无熔断基线调用: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06']
基线问题：连续失败也占用连接和重试预算，影响无关业务。


In [3]:
state = {'mode':'closed','failures':0,'cooldown':0}  # 初始化 closed/open/half-open 状态与失败计数。
def handle(row):  # 定义手写熔断和缓存降级状态机。
    if state['mode'] == 'open' and state['cooldown'] > 0:  # 判断是否仍在 open 冷却期。
        state['cooldown'] -= 1  # 消耗一个本地拒绝时间片。
        return 'fallback_cache:' + row['cache'], state['mode']  # 不调用下游并诚实返回带时间的缓存。
    if state['mode'] == 'open':  # 冷却结束后进入受限探测。
        state['mode'] = 'half_open'  # 只允许当前请求作为恢复探测。
    if row['response'] == 200:  # 处理成功的下游探测或正常调用。
        state['mode'] = 'closed'  # 成功后关闭熔断器恢复正常路径。
        state['failures'] = 0  # 清零连续基础设施失败次数。
        return 'live_success', state['mode']  # 返回真实新鲜数据成功状态。
    if row['response'] in {500,503}:  # 仅将依赖类 5xx 计入熔断窗口。
        state['failures'] += 1  # 累计连续基础设施失败次数。
        if state['failures'] >= 3:  # 达到阈值时立刻打开熔断器。
            state['mode'] = 'open'  # 切换到本地快速失败模式。
            state['cooldown'] = 1  # 设置一个教学用冷却时间片。
        return 'fallback_cache:' + row['cache'], state['mode']  # 失败时返回缓存降级而非伪造实时成功。
    return 'business_not_found', state['mode']  # 将 404 作为业务结果且不污染故障计数。

In [4]:
results = [(row['id'],) + handle(row) + (state['failures'],) for row in events]  # 按顺序驱动状态机并记录每步失败计数。
print('id | 动作 | 状态 | 失败计数')  # 输出熔断状态迁移表标题。
for item in results:  # 遍历六个请求对应的决策。
    print(item[0], item[1], item[2], item[3])  # 输出缓存降级、探测恢复和状态中间量。
print('实际下游调用近似数:', sum(not item[1].startswith('fallback_cache') for item in results))  # 汇总未被 open 状态短路的调用结果。

id | 动作 | 状态 | 失败计数
B01 fallback_cache:昨晚 22:00 已揽收 closed 1
B02 fallback_cache:昨晚 22:00 已揽收 closed 2
B03 fallback_cache:昨晚 22:00 已揽收 open 3
B04 fallback_cache:昨晚 22:00 已揽收 open 3
B05 live_success closed 0
B06 business_not_found closed 0
实际下游调用近似数: 2


In [5]:
bad_failures = 1  # 模拟错误地把 B06 的 404 也计为依赖失败。
correct_mode = results[-1][2]  # 读取正确状态机处理 B06 后的模式。
print('失败案例 B06：若把 404 计入故障则计数=', bad_failures, '，正确模式=', correct_mode)  # 展示业务失败不能触发全局熔断。
print('生产差距：生产版本需滑动窗口、按租户/工具分桶、真实时钟、半开并发上限、指标告警和写操作专用降级策略。')  # 说明教学状态机的生产扩展。

失败案例 B06：若把 404 计入故障则计数= 1 ，正确模式= closed
生产差距：生产版本需滑动窗口、按租户/工具分桶、真实时钟、半开并发上限、指标告警和写操作专用降级策略。


In [6]:
assert results[2][2] == 'open'  # 验证第三个连续 5xx 会打开熔断器。
assert results[3][1].startswith('fallback_cache')  # 验证 open 冷却期的请求不会触碰下游。
assert results[-1][1] == 'business_not_found'  # 验证 404 作为业务结果而不是被错误降级。